
# SVG → Centerline experiments

Цель: превратить залитую SVG-фигуру в компактную центральную линию.

Основной вариант здесь **полностью векторный**:

`SVG → svgelements → adaptive path sampling → Shapely Polygon → pygeoops.centerline`

Растеризация оставлена только как независимый baseline для сравнения.


In [ ]:

#@title 1. Install
!pip -q install svgelements shapely pygeoops matplotlib cairosvg scikit-image pillow


In [ ]:

#@title 2. Imports
from pathlib import Path
import io, math
import numpy as np
import matplotlib.pyplot as plt

from svgelements import SVG, Path as SvgPath, Shape, Move, Close
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union
import pygeoops

from google.colab import files
from IPython.display import display, SVG as DisplaySVG

WORK = Path("/content/centerline")
WORK.mkdir(exist_ok=True)


In [ ]:

#@title 3. Upload SVG
uploaded = files.upload()
name = next(n for n in uploaded if n.lower().endswith(".svg"))
svg_path = WORK / name
svg_path.write_bytes(uploaded[name])

display(DisplaySVG(filename=str(svg_path)))



## A. Vector SVG → Polygon

`svgelements` разбирает настоящий SVG: path-команды, Bézier, arc, transforms,
basic shapes и `<use>`.

Shapely не хранит кривые Bézier напрямую, поэтому кривую надо представить
достаточно точной полилинией. Это **не rasterization**: точки берутся прямо
с математической SVG-кривой.


In [ ]:

#@title 4. Parse filled SVG paths

svg = SVG.parse(str(svg_path), reify=True)

paths = []
for element in svg.elements():
    if isinstance(element, SvgPath):
        p = element
    elif isinstance(element, Shape):
        p = SvgPath(element)
        p.reify()
    else:
        continue

    if len(p) and str(getattr(element, "fill", "black")).lower() != "none":
        paths.append(p)

print("Filled vector paths:", len(paths))
for i, p in enumerate(paths):
    print(i, "segments:", len(p), "bbox:", p.bbox())


In [ ]:

#@title 5. Adaptive-ish vector sampling

# Максимальная длина прямого кусочка полилинии в SVG units.
# Для GClef с width≈15 значение 0.02 уже очень плотное.
MAX_STEP = 0.02  #@param {type:"number"}

def sample_segment(seg, max_step):
    length = max(float(seg.length(error=1e-6)), 1e-9)
    n = max(2, int(math.ceil(length / max_step)) + 1)
    return [(float(seg.point(t).x), float(seg.point(t).y))
            for t in np.linspace(0.0, 1.0, n)]

def path_subpaths_to_rings(path, max_step):
    rings, current = [], []

    for seg in path:
        if isinstance(seg, Move):
            if len(current) >= 3:
                rings.append(current)
            current = [(float(seg.end.x), float(seg.end.y))]
            continue

        pts = sample_segment(seg, max_step)

        if not current:
            current.append(pts[0])

        current.extend(pts[1:])

        if isinstance(seg, Close):
            if len(current) >= 3:
                if current[0] != current[-1]:
                    current.append(current[0])
                rings.append(current)
            current = []

    if len(current) >= 3:
        if current[0] != current[-1]:
            current.append(current[0])
        rings.append(current)

    return rings

all_rings = []
for path in paths:
    all_rings.extend(path_subpaths_to_rings(path, MAX_STEP))

print("Closed rings:", len(all_rings))
print("Points:", [len(r) for r in all_rings])


In [ ]:

#@title 6. Rings → filled Shapely geometry

# Для compound SVG path с nonzero fill rule удобно собрать кольца
# через symmetric_difference. Для типовых глифов это корректно сохраняет
# внешние области и отверстия независимо от порядка subpath-ов.

geometry = None
for ring in all_rings:
    poly = Polygon(ring)
    if not poly.is_valid:
        poly = poly.buffer(0)
    if poly.is_empty:
        continue
    geometry = poly if geometry is None else geometry.symmetric_difference(poly)

if geometry is None:
    raise ValueError("No filled polygon produced")

geometry = geometry.buffer(0)

print("Geometry:", geometry.geom_type)
print("Area:", geometry.area)
print("Bounds:", geometry.bounds)

fig, ax = plt.subplots(figsize=(7, 12))
polys = [geometry] if geometry.geom_type == "Polygon" else list(geometry.geoms)
for p in polys:
    x, y = p.exterior.xy
    ax.plot(x, y)
    for hole in p.interiors:
        x, y = hole.xy
        ax.plot(x, y)
ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_title("Vector-derived polygon")
plt.show()



> Примечание: для общего SVG позже стоит уважать `fill-rule="nonzero|evenodd"`
> буквально. Для нашего текущего глифа compound path простой; эта ячейка
> намеренно минимальна для эксперимента.


In [ ]:

#@title 7. pygeoops vector centerline

DENSIFY_DISTANCE = -1.0    #@param {type:"number"}
MIN_BRANCH_LENGTH = -1.0   #@param {type:"number"}
SIMPLIFY = -0.25           #@param {type:"number"}

centerline = pygeoops.centerline(
    geometry,
    densify_distance=DENSIFY_DISTANCE,
    min_branch_length=MIN_BRANCH_LENGTH,
    simplifytolerance=SIMPLIFY,
)

print(centerline.geom_type, "length =", centerline.length)

fig, ax = plt.subplots(figsize=(7, 12))
for p in polys:
    x, y = p.exterior.xy
    ax.plot(x, y, linewidth=0.7)

lines = [centerline] if centerline.geom_type == "LineString" else list(centerline.geoms)
for line in lines:
    x, y = line.xy
    ax.plot(x, y, linewidth=1.5)

ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_title("pygeoops centerline from SVG vectors")
plt.show()



## B. Raster medial axis — только baseline

Этот вариант оставляем, потому что `medial_axis` дополнительно выдаёт
distance-to-boundary (локальную полуширину ленты), что может пригодиться
для сравнения и дальнейших эвристик. Но он больше не нужен для получения
полигона для `pygeoops`.


In [ ]:

#@title 8. Optional raster baseline
import cairosvg
from PIL import Image
from skimage.morphology import medial_axis

SCALE = 12  #@param {type:"integer"}

png = cairosvg.svg2png(
    bytestring=svg_path.read_bytes(),
    scale=SCALE,
    background_color="white"
)
gray = np.array(Image.open(io.BytesIO(png)).convert("L"))
mask = gray < 245

skeleton, distance = medial_axis(mask, return_distance=True)
ys, xs = np.nonzero(skeleton)

plt.figure(figsize=(7, 12))
plt.imshow(mask, cmap="gray")
plt.scatter(xs, ys, s=.25)
plt.axis("off")
plt.title("Raster medial axis baseline")
plt.show()

r = distance[skeleton]
print("skeleton pixels:", len(r))
print("width median:", float(np.median(r) * 2))



## Что дальше

Если векторный `pygeoops.centerline` даёт разумный сырой skeleton, следующий
этап — превратить его в граф и минимизировать лишние ветки:

1. junction/end-point detection;
2. pruning коротких spur-веток;
3. pairing рёбер в junction по tangent + curvature + width continuity;
4. сборка длинных непрерывных линий;
5. simplification и cubic Bézier fitting;
6. нормализация в безразмерный fingerprint.
